# IQ-`.dat` einlesen und alle Frequenzslots abhören

Das Format ist bekannt und wird hier **nicht** mehr diagnostiziert:

| | |
|---|---|
| Datentyp | `<i2` (int16, little-endian), je Element Real- dann Imaginärteil |
| Achsenreihenfolge | `[time][freq][antenna]` — die Antenne läuft am schnellsten |
| Frequenzslots | 16 |
| ADC-Kanäle | 8 (7 Antennen + 1 unbelegt) |
| Abtastrate | 152 kHz |
| Header | keiner |

Ein Zeitschritt belegt damit $16\cdot8\cdot2\cdot2 = 512$ Byte, die Adresse eines
Elements ist $\text{offset}(t,f,a) = ((t\cdot16 + f)\cdot8 + a)\cdot4$.

Das Notebook liest die Datei als `memmap`, demoduliert jeden der 16 Slots zu Mono-Audio
und spielt ihn ab. Klingen alle 16 nach Programm, ist das Speicherlayout über die
gesamte Slot-Achse korrekt — ein Stride-Fehler bliebe nicht auf einen Slot beschränkt.

**Grenze der Methode:** Ein Vertauschen von I und Q ergibt $z' = j\,z^*$, also
$m'(t) = -m(t)$ — akustisch nicht unterscheidbar. Für das Radar kehrt das jedoch das
Vorzeichen von Doppler und Einfallswinkel um. Die Hörprobe erfasst Achsenreihenfolge,
Stride, Endianness und Skalierung, **nicht** die Konjugation.

In [ ]:
import os
from fractions import Fraction

import numpy as np
from scipy.signal import bilinear, filtfilt, firwin, lfilter, resample_poly
from IPython.display import Audio, display

# ---- Datei -----------------------------------------------------------------
DAT_PATH = "target.dat"

# ---- Format ------------------------------------------------------
N_ANTENNAS   = 8                  # ADC-Kanaele je Zeitschritt
N_FREQ       = 16                 # Frequenzslots
DTYPE        = np.dtype("<i2")    # int16, little-endian
HEADER_BYTES = 0                  # Layout [time][freq][antenna]

# ---- Signal ----------------------------------------------------------------
FS_HZ        = 152_000.0
DEVIATION_HZ = 75_000.0           # Spitzenhub, normgerecht +-75 kHz

# ---- Hoerprobe -------------------------------------------------------------
ANTENNA   = 0        # fuers Audio beliebig: alle Elemente tragen dasselbe Programm
T_START_S = 0.0
SNIPPET_S = 3.0      # Laenge des Ausschnitts je Slot

# ---- Audio -----------------------------------------------------------------
FS_AUDIO       = 48_000
USE_DEEMPHASIS = True     # reale Aufnahme: True, eigene Erzeugung: False
TAU_S          = 50e-6
DESPIKE        = True     # Ueberlaufklicks des Diskriminators interpolieren

print(f"NumPy {np.__version__}")

## 1. Datei öffnen

`memmap` statt `fromfile`: die Datei ist 2 GiB groß, gebraucht wird je Hörprobe aber nur
ein Ausschnitt von wenigen Sekunden aus einem von 128 Kanälen.

**DC-Abzug.** Ein konstanter Offset (LO-Leck, ADC-Offset) verschiebt die Ortskurve aus
dem Ursprung. Der Phasendiskriminator liest das als mit der Signalphase modulierten
Frequenzfehler — hörbar als Brummen und Verzerrung. Der Mittelwert wird deshalb vor der
Demodulation abgezogen.

**Keine Rückskalierung.** Der Faktor `scale` aus dem eigenen Writer ist bei einer
Fremddatei unbekannt und für die FM-Demodulation irrelevant: die Information steckt in
der Phase, nicht in der Amplitude. Gelesen wird in Rohzählwerten.

In [ ]:
BYTES_PER_STEP = N_FREQ * N_ANTENNAS * 2 * DTYPE.itemsize
file_size = os.path.getsize(DAT_PATH)
T_TOTAL, rest = divmod(file_size - HEADER_BYTES, BYTES_PER_STEP)
if rest:
    raise ValueError(f"{rest} B Rest -- Datei passt nicht zu {N_FREQ} Slots "
                     f"x {N_ANTENNAS} Kanaelen x {DTYPE.itemsize} B")

mm = np.memmap(DAT_PATH, dtype=DTYPE, mode="r", offset=HEADER_BYTES,
               shape=(T_TOTAL, N_FREQ, N_ANTENNAS, 2))


def get_channel(freq_index, antenna, i0, i1):
    """Einen (Slot, Antenne)-Kanal als complex128 herausschneiden, DC abgezogen."""
    blk = np.asarray(mm[i0:i1, freq_index, antenna, :], dtype=np.float64)
    z = blk[:, 0] + 1j * blk[:, 1]
    return z - z.mean()


print(f"{file_size:,} B = {file_size/2**20:.1f} MiB, {BYTES_PER_STEP} B/Zeitschritt")
print(f"{T_TOTAL:,} Zeitschritte = {T_TOTAL/FS_HZ:.2f} s bei {FS_HZ/1e3:.0f} kHz")
print(f"memmap {mm.shape}")

## 2. FM-Demodulation

$$\hat f[n] = \frac{f_s}{2\pi}\arg\!\left(z[n]\,z^*[n-1]\right),
\qquad m[n] = \hat f[n]/\Delta f$$

Der Diskriminator ist eindeutig, solange der Phasenschritt betragsmäßig unter $\pi$
bleibt, also $|f(t)| < f_s/2$. Bei $f_s = 152$ kHz liegt die Grenze bei 76 kHz gegen
einen Spitzenhub von 75 kHz — **rund 1,3 % Reserve**. Rauschspitzen überschreiten sie
gelegentlich, der Diskriminator springt dort um $2\pi$, hörbar als Klick. Das ist keine
Eigenschaft der Datei, sondern der Kombination aus Kanalisierungsrate und Hub.
`DESPIKE` ersetzt diese isolierten Ausreißer durch lineare Interpolation der
Nachbarwerte — das Nutzsignal ist bandbegrenzt und damit glatt, ein Sprung um $2\pi$
ist es nicht. Reine Kosmetik für die Hörprobe.

Danach Monosignal per Tiefpass 0–15 kHz (der Faktor $1/0{,}9$ macht den Pegelanteil des
Pilottons rückgängig), Ratenwandlung auf 48 kHz und optional die Deemphase: reale Sender
preemphasieren mit 50 µs, ohne Gegenfilter klingen die Höhen zischend. Für Dateien aus
dem eigenen Sendemodell, das keine Preemphase enthält, `USE_DEEMPHASIS = False`.

In [ ]:
LP_MONO = firwin(301, 15_000, fs=FS_HZ)
RATIO = Fraction(int(FS_AUDIO), int(round(FS_HZ))).limit_denominator(4000)


def deemphasis(x, fs, tau=TAU_S):
    b, a = bilinear([1.0], [tau, 1.0], fs=fs)
    return lfilter(b, a, x)


def slot_audio(freq_index, antenna=ANTENNA, t_start_s=T_START_S, len_s=SNIPPET_S):
    """Einen Frequenzslot zu Mono-Audio bei FS_AUDIO demodulieren."""
    i0 = int(round(t_start_s * FS_HZ))
    i1 = min(i0 + int(round(len_s * FS_HZ)), T_TOTAL)
    z = get_channel(freq_index, antenna, i0, i1)

    dphi = np.angle(z[1:] * np.conj(z[:-1]))
    mpx = dphi * FS_HZ / (2 * np.pi) / DEVIATION_HZ

    n_spikes = 0
    if DESPIKE:
        bad = np.abs(dphi) > 0.98 * np.pi
        n_spikes = int(bad.sum())
        if n_spikes:
            idx, ok = np.flatnonzero(bad), np.flatnonzero(~bad)
            mpx[idx] = np.interp(idx, ok, mpx[ok])

    mono = filtfilt(LP_MONO, [1.0], mpx) / 0.9
    audio = resample_poly(mono, RATIO.numerator, RATIO.denominator)
    if USE_DEEMPHASIS:
        audio = deemphasis(audio, FS_AUDIO)
    return audio, n_spikes


print(f"Ratenwandlung {FS_HZ/1e3:.0f} kHz -> {FS_AUDIO/1e3:.0f} kHz "
      f"als {RATIO.numerator}/{RATIO.denominator}")

## 3. Alle Frequenzslots abhören

Je Slot ein Ausschnitt von `SNIPPET_S` Sekunden ab `T_START_S`, gelesen von Antenne
`ANTENNA`. `normalize=True` gleicht die unterschiedlichen Empfangspegel aus, sonst wären
die schwachen Slots kaum hörbar.

In [ ]:
for k in range(N_FREQ):
    audio, n_spikes = slot_audio(k)
    print(f"Slot {k:2d}   {len(audio)/FS_AUDIO:.2f} s   "
          f"Klicks {n_spikes} ({100*n_spikes*RATIO.denominator/RATIO.numerator/max(len(audio),1):.4f} %)")
    display(Audio(audio, rate=FS_AUDIO, normalize=True))

### Einzelner Slot, längerer Ausschnitt

Zum genaueren Hinhören: ein Slot über eine frei wählbare Dauer.

In [ ]:
SLOT = 12          # Slot-Nummer
LEN_S = 15.0       # Dauer in Sekunden

audio, n_spikes = slot_audio(SLOT, len_s=LEN_S)
print(f"Slot {SLOT}, Antenne {ANTENNA}, {len(audio)/FS_AUDIO:.2f} s, "
      f"{n_spikes} Klicks interpoliert")
display(Audio(audio, rate=FS_AUDIO, normalize=True))